# 🎓 Day 1 · Session 5
# 05B · Build Your First RAG Chatbot
## Build a complete retrieval-and-generation pipeline with citations

> **Teaching flow:** Why → What → How → When to use

## 🎯 Learning objectives

- Understand the concept clearly
- Run a simple, reliable demonstration
- Connect the code to the RAG architecture shown in the slides
- Recognise limitations and production considerations

## 🔧 Setup

Create a `.env` file in the notebook folder:

```env
OPENAI_API_KEY=your-key-here
```

The notebook also has an offline fallback so every cell can execute without an API key.

In [ ]:
# Run once if packages are missing
#%pip install scikit-learn
#%pip install -q -U openai python-dotenv numpy pandas scikit-learn

/Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

load_dotenv(Path.cwd() / '.env', override=False)
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key) if (OpenAI is not None and api_key) else None

CHAT_MODEL = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')

if OpenAI is None:
    print('OpenAI package not installed. Run the installation cell for live API mode.')
print('OpenAI mode:', 'LIVE API' if client else 'OFFLINE FALLBACK')
print('Chat model:', CHAT_MODEL)
print('Embedding model:', EMBEDDING_MODEL)


OpenAI mode: LIVE API
Chat model: gpt-4o-mini
Embedding model: text-embedding-3-small


In [7]:
def generate_answer(prompt, instructions='You are a helpful academic assistant.'):
    if client is None:
        return '[Offline mode] Add OPENAI_API_KEY to run the live LLM answer.'
    response = client.responses.create(
        model=CHAT_MODEL,
        instructions=instructions,
        input=prompt,
        max_output_tokens=500,
    )
    return response.output_text

def openai_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return np.array(response.data[0].embedding, dtype=float)

def cosine_score(a, b):
    a, b = np.asarray(a), np.asarray(b)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denominator) if denominator else 0.0

## 1️⃣ Knowledge base

In [8]:
sample_docs = [
 {'source':'department_handbook.txt','text':'M.Tech CSE tuition is Rs. 50,000 per semester. The programme has 4 semesters. GATE-qualified students receive a 50% tuition scholarship. Students need 75% attendance for final examinations.'},
 {'source':'course_catalog.txt','text':'The NLP elective is offered in Semester 3. Prerequisites are Machine Learning and Python Programming. The faculty coordinator is Dr. Kavitha Raman. Deep Learning is offered in Semester 4.'},
 {'source':'lab_manual.txt','text':'AI Lab topics include Python, preprocessing, regression, classification, neural networks and evaluation. Students must submit lab records every Friday.'}
]
pd.DataFrame(sample_docs)

,source,text
0,department_handbook.txt,"M.Tech CSE tuition is Rs. 50,000 per semester...."
1,course_catalog.txt,The NLP elective is offered in Semester 3. Pre...
2,lab_manual.txt,"AI Lab topics include Python, preprocessing, r..."


## 2️⃣ Chunk documents and preserve metadata

In [9]:
def chunk_documents(docs, chunk_size=240, overlap=40):
    if overlap >= chunk_size:
        raise ValueError('overlap must be smaller than chunk_size')
    output=[]
    for doc in docs:
        text=doc['text'].strip(); start=0
        while start < len(text):
            end=min(start+chunk_size, len(text))
            output.append({'text':text[start:end], 'source':doc['source'], 'start':start, 'end':end})
            if end == len(text): break
            start=end-overlap
    return output

chunks=chunk_documents(sample_docs)
print('Documents:',len(sample_docs),'Chunks:',len(chunks))
pd.DataFrame(chunks)

Documents: 3 Chunks: 3


,text,source,start,end
0,"M.Tech CSE tuition is Rs. 50,000 per semester....",department_handbook.txt,0,189
1,The NLP elective is offered in Semester 3. Pre...,course_catalog.txt,0,187
2,"AI Lab topics include Python, preprocessing, r...",lab_manual.txt,0,151


## 3️⃣ Build the vector index

For the core FDP demo we keep the implementation transparent. OpenAI embeddings are used when an API key is available; otherwise the notebook uses a local TF-IDF fallback.

In [10]:
if client:
    for chunk in chunks:
        chunk['vector'] = openai_embedding(chunk['text'])
    INDEX_MODE='OpenAI embeddings'
    vectorizer=None
else:
    vectorizer=TfidfVectorizer(ngram_range=(1,2))
    matrix=vectorizer.fit_transform([c['text'] for c in chunks])
    for i,chunk in enumerate(chunks):
        chunk['vector']=matrix[i]
    INDEX_MODE='offline TF-IDF fallback'
print('Index mode:',INDEX_MODE)

Index mode: OpenAI embeddings


## 4️⃣ Retriever

In [11]:
def retrieve(question, top_k=3, min_score=None):
    if client:
        qv=openai_embedding(question)
        scored=[{**{k:v for k,v in c.items() if k!='vector'}, 'score':cosine_score(qv,c['vector'])} for c in chunks]
    else:
        qv=vectorizer.transform([question])
        scored=[{**{k:v for k,v in c.items() if k!='vector'}, 'score':float(sklearn_cosine_similarity(qv,c['vector'])[0,0])} for c in chunks]
    scored=sorted(scored,key=lambda x:x['score'],reverse=True)
    if min_score is not None:
        scored=[x for x in scored if x['score']>=min_score]
    return scored[:top_k]

retrieved=retrieve('What are the prerequisites for NLP?')
pd.DataFrame(retrieved)

,text,source,start,end,score
0,The NLP elective is offered in Semester 3. Pre...,course_catalog.txt,0,187,0.592602
1,"AI Lab topics include Python, preprocessing, r...",lab_manual.txt,0,151,0.342393
2,"M.Tech CSE tuition is Rs. 50,000 per semester....",department_handbook.txt,0,189,0.126051


## 5️⃣ Grounded answer with source citation

In [12]:
def rag_answer(question, top_k=3):
    retrieved=retrieve(question,top_k=top_k)
    context='\n\n'.join([f"SOURCE: {x['source']}\nCONTENT: {x['text']}" for x in retrieved])
    prompt=f'''Answer ONLY from the context.
If the answer is absent, reply exactly: "I don't have this information in the provided knowledge base."
Do not guess. End a supported answer with a Sources line listing only the source filenames used.

CONTEXT:
{context}

QUESTION:
{question}'''
    if client:
        answer=generate_answer(prompt,'You are a cautious academic knowledge assistant.')
    else:
        answer='[Offline retrieval completed. Add OPENAI_API_KEY for generation.]'
    return {'question':question,'answer':answer,'retrieved_chunks':retrieved}

result=rag_answer('What are the prerequisites for NLP?')
print(result['answer'])
pd.DataFrame(result['retrieved_chunks'])

The prerequisites for NLP are Machine Learning and Python Programming. 

Sources: course_catalog.txt


,text,source,start,end,score
0,The NLP elective is offered in Semester 3. Pre...,course_catalog.txt,0,187,0.592602
1,"AI Lab topics include Python, preprocessing, r...",lab_manual.txt,0,151,0.342393
2,"M.Tech CSE tuition is Rs. 50,000 per semester....",department_handbook.txt,0,189,0.126051


## 6️⃣ In-scope and out-of-scope test

In [13]:
test_questions=['What is the M.Tech CSE fee?','Who coordinates NLP?','When are lab records submitted?','What is the hostel fee?']
for q in test_questions:
    r=rag_answer(q)
    print()
    print('QUESTION:',q)
    print('ANSWER:',r['answer'])
    print('TOP SOURCE:',r['retrieved_chunks'][0]['source'] if r['retrieved_chunks'] else 'None')



QUESTION: What is the M.Tech CSE fee?
ANSWER: The M.Tech CSE tuition is Rs. 50,000 per semester. Since the programme has 4 semesters, the total fee would be Rs. 200,000. 

Sources: department_handbook.txt
TOP SOURCE: department_handbook.txt

QUESTION: Who coordinates NLP?
ANSWER: Dr. Kavitha Raman coordinates NLP. 

Sources: course_catalog.txt
TOP SOURCE: course_catalog.txt

QUESTION: When are lab records submitted?
ANSWER: Lab records are submitted every Friday. 

Sources: lab_manual.txt
TOP SOURCE: lab_manual.txt

QUESTION: What is the hostel fee?
ANSWER: I don't have this information in the provided knowledge base.
TOP SOURCE: department_handbook.txt


## 7️⃣ Vector store mapping

The transparent in-memory list above plays the role of a tiny vector store. In a real application, replace it with **Chroma**, **FAISS**, or a managed vector database. The surrounding RAG logic remains the same.

In [16]:
RUN_OPTIONAL_CHROMA=False
if RUN_OPTIONAL_CHROMA:
    try:
        from langchain_chroma import Chroma
        from langchain_openai import OpenAIEmbeddings
        from langchain_core.documents import Document
        if not client:
            raise RuntimeError('OPENAI_API_KEY is required for this optional demo.')
        docs=[Document(page_content=c['text'],metadata={'source':c['source']}) for c in chunks]
        store=Chroma.from_documents(docs,OpenAIEmbeddings(model=EMBEDDING_MODEL),persist_directory='./chroma_rag_demo')
        found=store.similarity_search('What are the prerequisites for NLP?',k=3)
        for d in found: print(d.metadata,d.page_content)
    except ImportError:
        print('Install optional packages: %pip install -q -U langchain-core langchain-openai langchain-chroma chromadb')
else:
    print('Optional Chroma demo skipped. Core RAG pipeline is complete.')

Optional Chroma demo skipped. Core RAG pipeline is complete.


## ⭐ Wow moment

Change the user's wording completely—e.g., **'What background knowledge is needed before choosing the language elective?'**—and inspect whether the same NLP chunk is retrieved.

## ✅ Takeaways

- A working RAG system has an index, retriever, grounded prompt and generator.
- Keep source metadata with every chunk.
- Inspect retrieved chunks before blaming the LLM.
- Vector databases scale storage and retrieval; they do not replace evaluation or access control.